In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# Installing pyspark and spark-nlp
! pip install --upgrade -q pyspark==3.3.0 spark-nlp==4.2.8

In [3]:
import json
import pandas as pd
import numpy as np

import sparknlp
import pyspark.sql.functions as F

from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from sparknlp.annotator import *
from sparknlp.base import *
from sparknlp.pretrained import PretrainedPipeline
from pyspark.sql.types import StringType, IntegerType

# Start PySpark Session

In [4]:
spark = sparknlp.start()

print("Spark NLP version", sparknlp.version())
print("Apache Spark version:", spark.version)

spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-113d7bcc-fa8e-4efc-a37c-e999ab76d8e4;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;4.2.8 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.828 in central
	found com.github.universal-automata#liblevenshtein;3.0.0 in central
	found com.google.protobuf#protobuf-java-util;3.0.0-beta-3 in central
	found com.google.protobuf#protobuf-java;3.0.0-beta-3 in central
	found com.google.code.gson#gson;2.3 in central
	found it.unimi.dsi#fastutil;7.0.12 in central
	found org.projectlombok#lombok;1.16.8 in central
	found com.google.cloud#google-cloud-storage;2.15.0 in central
	found com.google.guava#guava;31.1-jre in central
	found com.google.guava#failureaccess;1.0.1 

26/04/17 18:49:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark NLP version 4.2.8
Apache Spark version: 3.3.0


# Select DL Model

In [5]:
### Select Model
model_antisemitism = 'bert_sequence_classifier_antisemitism'
model_trec_coarse = "bert_sequence_classifier_trec_coarse"
model_age_news = "bert_sequence_classifier_age_news"
model_hatexplain = "bert_sequence_classifier_hatexplain"
model_emotion = "bert_sequence_classifier_emotion"
model_banking = "bert_sequence_classifier_banking77"

# Samples

In [6]:
text_antisemitism = ["""Shylock in Merchant of Venice. Shylock was a Jew and moneylender. Depends on the context it is used but as the antisemitism is hotly debated nowadays if I were a Jew I wouldn't like to hear it. Perhaps I'm wrong but that's my opinion.""",
"""That Jew gripped yo nuts and you did nothing and you been attacking black people ever since. Probably like that shit""",
"""They came for the Jews, and I did not speak out Because I was not a Jew.Then they came for me and there was no one left to speak for me""",
"""David is a sephardic jew huh.... now i have to give him my entire heart i guess""",
"""I asked a genuine question, she has been smearing @georgegalloway for a while now without any evidence. Am I not allowed to ask her to show me the anti Semitism. Remember @RachelRileyRR is the one who said she ‘doesn’t look like a normal jew’ that to me is anti Semitic""",
"""I pointed the finger directly at the fascists still in control of Europe the muh jew shills began in earnest. Distraction, anger, insult, lies.""",
]

In [7]:
text_trec_coarse = ["""Germany is the largest country in Europe economically.""",
"""What other prince showed his paintings in a two-prince exhibition with Prince Charles in London?""",
"""What is the name of the chronic neurological autoimmune disease which attacks the protein sheath that surrounds nerve cells causing a gradual loss of movement in the body?""",
"""How many hands does Bjorn Borg use when hitting his forehand?""",
"""CNN is the abbreviation for what?""",
"""Give a reason for American Indians oftentimes dropping out of school.""",
"""What was organized as a confederate veterans social club in Pulaski, in Tennessee, in 1866?""",
"""Who was the first person inducted into the U.S. Swimming Hall of Fame?""",
"""What did the only repealed amendment to the U.S. Constitution deal with?""",
]

In [8]:
text_age_news = ["""Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group, which has a reputation for making well-timed and occasionally controversial plays in the defense industry, has quietly placed its bets on another part of the market.""",
"""Juventus and Olympique Lyon joined Chelsea and Inter Milan in the last 16 of the Champions League after both scored late goals to seal their places on Wednesday.""",
"""Fearing the fate of Italy, the centre-right government has threatened to be merciless with those who flout tough restrictions.""",
"""Cisco Systems is teaming up with researchers at the Massachusetts Institute of Technology to develop new communications technology that the company hopes to ultimately turn into products.""",
"""Golf is not a sport renowned for being at ease with gender issues. Many of the games great clubs refuse to admit women, and the occasional inclusion of women in men professional tournaments has caused consternation from Arran to Augusta.""",
"""The Supreme Council of the United Arab Emirates (UAE) on Wednesday evening unanimously elected Sheikh Khalifa bin Zayed Al Nahyan as the new president of the UAE, the official Emirates news agency WAM reported Wednesday.""",
"""The Norwegian hacker famed for developing DVD encryption cracking software has apparently struck again; this time breaking the locks on Apple Computer Inc.'s wireless music streaming technology.""",
"""Utility holding company Ameren Corp. announced plans Wednesday to reduce its payroll by offering 950 union and management employees financial incentives to leave."""
]

In [9]:
text_hatexplain = ["""If English is not imposition then Hindi is also not imposition shut up chutiya retards stop hindi imposition.""",
"""The uk has threatened to return radioactive waste to the eu if an agreement cannot be reached can not we keep the radioactive waste and send back all the paki migrants instead""",
"""Afghani immigrant once told me that in Afghanistan we do not believe in fences while our conversation was about livestock guardian dogs it now occurs to me what this really means is that there no belief in protection of persons or property""",
"""I post a video showing the shocking promotion of violence by the south african eff themselves and jewtube still say i am promoting violence""",
"""I want to go back to school and get an education but i am poor and colleges are kike rip offs i am not satisfied with where i am in life and honestly have no clue where or how to start pushing forward""",
"""I am confused how a president blocked stimulus checks for those who married to immigrants while he himself married to an immigrant""",
"""I love you very much!"""]

In [10]:
text_emotion = ["""Dr Priya Patel in the respiratory clinic was absolutely wonderful. She took time to explain my COPD management plan and made sure I understood every step. The nurse on reception, Karen Thompson, was also very welcoming.""",
            """The parking at 45 Albert Street is very limited. I had to walk from the Westfield car park which is difficult at my age.""",
            """I had my appointment on 15 January 2026 at 2pm and wasn't seen until after 3:15pm. Dr James Richardson seemed rushed and didn't listen to my concerns about the medication side effects. I called the clinic at 07 3456 7890 twice before my appointment to ask about preparation and nobody answered. My wife Angela Chen had a much better experience at the Chermside Day Surgery last month.""",
            """The physiotherapist Michael was great. He gave me a detailed home exercise program after my knee surgery and followed up via email at susan.obrien82@gmail.com to check on my progress. Very impressed with that level of care.""",
            """The online booking system could be easier to use. I ended up calling reception to book because I couldn't figure out the Zedoc portal. I was referred by Dr Nguyen at the Ipswich Hospital and the transition was smooth.""",
            """Nurse Rebecca Taylor in the diabetes clinic was exceptional. She spent over 40 minutes with me going through my HbA1c results and adjusting my insulin plan. She also coordinated with my GP Dr Samantha Lee at the Toowoomba Medical Practice to ensure consistent care.""",
            """Dr Patel is always thorough and patient. She remembers details from previous visits which makes me feel valued.""",
            """I'm 81 years old and find the forms quite difficult to fill in. My daughter Sarah Fitzpatrick usually helps me but she wasn't available this time. Could you offer some assistance at the front desk for elderly patients? Also my Medicare number is 2345 67890 1 and I think there was a billing error on my last visit on 28 January 2026.""",   
            """The midwifery team, especially Lisa and Jenny, were amazing throughout my pregnancy. They made me feel safe and supported. The antenatal classes at the centre on George Street were also excellent.""",      
            """It would be nice to have later appointment slots. As a working mum I find it hard to attend before 4pm. My employer at Bright Horizons Childcare is not always flexible with time off. I recommended this clinic to my sister-in-law Patricia Nakamura who is expecting in July.""",
            """The new blood collection nurse, Daniel Kim, was very skilled. Best blood draw I've had - barely felt it. He mentioned he previously worked at the Royal Brisbane and Women's Hospital.""",
            """The results portal is confusing. I had to call Dr Richardson's office at 07 3456 7891 to get my pathology explained because I couldn't understand the online report.""",    
            """Dr Patel and Nurse Rebecca were both excellent. They were respectful of my cultural needs and ensured a female practitioner was available for my examination. This was very important to me.""",
            """The interpreter service was not available on 4 March 2026 when my mother attended. She speaks Arabic and struggled to communicate her symptoms. Please ensure interpreters are booked in advance. My mother Zahra Al-Rashid (patient ID PAT-91603) would like to provide feedback separately. Can someone contact her at fatima.alrashid@outlook.com to arrange an Arabic-language survey?""",
            """Outstanding mental health support from psychologist Dr Amanda Clarke. She helped me develop coping strategies after my workplace incident at BHP Mitsubishi Alliance in Mount Isa last year. The telehealth option made it possible to continue sessions when I was FIFO.""",
            """The mental health waiting list is too long. I was referred on 15 November 2025 and didn't get my first appointment until 8 January 2026. That's nearly 8 weeks.""",   
            """The wound care nurse Jacinta was thorough and gentle. She explained the healing process for my post-surgical wound clearly and gave me written instructions to take home.""",
            """I received a reminder SMS from Zedoc to complete this survey but the link didn't work on my Samsung phone. I had to use my husband David Tran's iPhone instead. The SMS came from number 0437 123 456. I was transferred from Logan Hospital after my surgery there on 2 March 2026. The handover between hospitals could have been smoother - my medication list wasn't updated correctly.""",
            """The entire cardiology team is first-rate. Dr Richardson personally called me at home on 0478 234 567 to discuss my stress test results. That kind of proactive communication is rare and very reassuring.""",
            """The car park needs more disabled bays. I have a temporary mobility permit after my cardiac rehab and struggled to find a spot on 20 March 2026. My cardiologist at the Prince Charles Hospital, Dr Andrew Walsh, also coordinates with Dr Richardson here which gives me great confidence in my care plan."""]

In [11]:
text_banking = ["""I have been waiting over a week. Is the card still coming?""",
"""I need a transaction reversed from my account.""",
"""How long does it take for cards to be delivered after ordering them?""",
"""I've just been married and need to update my name""",
"""I'm interested in learning more about disposable virtual cards.	""",
"""I tried topping up using my card, but the money is gone?"""]

# Define Spark NLP pipeline

In [12]:
model_dict = {model_antisemitism: text_antisemitism,
              model_trec_coarse :text_trec_coarse,
              model_age_news :text_age_news,
              model_hatexplain: text_hatexplain,
              model_emotion: text_emotion,
              model_banking: text_banking}

In [13]:
import pandas as pd
import os
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from sparknlp.annotator import *
from sparknlp.base import *
from pyspark.sql import functions as F

# Initialize results dictionary
results = {}

def run_pipeline(model_name, text_data, results_dict):
    """
    Complete Spark NLP Pipeline: Stages data, initializes all annotators,
    and caches results to bypass Python 3.12 compatibility issues.
    """
    """
    Stages data via CSV to bypass Python 3.12 pickling errors in PySpark 3.3.
    Includes .cache() and eager execution to prevent FileNotFound errors.
    """
    print(f"\n" + "="*60)
    print(f"INITIATING PIPELINE FOR: {model_name}")
    print("="*60)

    # Define a unique temporary filename for this model run
    # Replacing slashes to prevent directory errors in the filename
    clean_name = model_name.replace("/", "_")
    temp_filename = f"staged_data_{clean_name}.csv"
    
    try:
        # 1. DEFINE BASE ANNOTATORS (Missing parts added back here)
        document_assembler = DocumentAssembler() \
            .setInputCol('text') \
            .setOutputCol('document')

        tokenizer = Tokenizer() \
            .setInputCols(['document']) \
            .setOutputCol('token')

        # 2. DEFINE THE DYNAMIC CLASSIFIER
        sequenceClassifier = BertForSequenceClassification \
            .pretrained(model_name, 'en') \
            .setInputCols(['token', 'document']) \
            .setOutputCol('pred_class')

        # 3. ASSEMBLE THE COMPLETE PIPELINE
        nlpPipeline = Pipeline(stages=[
            document_assembler, 
            tokenizer, 
            sequenceClassifier
        ])

        # 4. STAGE DATA TO CSV
        # Convert input to a DataFrame and save to local disk
        if isinstance(text_data, str):
            text_data = [text_data]
        pd.DataFrame(text_data, columns=["text"]).to_csv(temp_filename, index=False)
        
        # 5. LOAD INTO SPARK AND CACHE EAGERLY
        # We cache the raw DF and the result to ensure memory persistence
        # Use Spark's native Java reader and CACHE the initial DataFrame
        df = spark.read.option("header", "True").option("inferSchema", "True").csv(temp_filename).cache()
        
        # 6. RUN THE PIPELINE AND TRIGGER MATERIALIZATION
        # Fit and Transform
        # We CACHE the result so it stays in memory after the CSV is deleted
        processed_df = nlpPipeline.fit(df).transform(df).cache()
        
        # This .count() is the 'Action' that saves the day in Python 3.12
        # EAGER EXECUTION TRIGGER
        # We must call an 'action' (like .count()) while the CSV still exists.
        # This forces Spark to read the file and store the result in the cache.
        record_count = processed_df.count()
        
        # Store in results
        # Store the result in your dictionary
        results_dict[model_name] = processed_df
        print(f"Inference complete. {record_count} records materialized in Spark memory.")

        # 7. Show a preview of the results
        print(f"Success! {record_count} rows cached in memory for {model_name}.")
        # Use the specific output column from your classifier (e.g., 'pred_class' or 'class')
        if "pred_class" in processed_df.columns:
            processed_df.select("text", "pred_class.result").show(20, truncate=False)
        elif "class" in processed_df.columns:
            processed_df.select("text", "class.result").show(20, truncate=False)

    except Exception as e:
        print(f"CRITICAL ERROR in {model_name}: {str(e)}")
        
    finally:
        # Cleanup the physical file
        # Clean up the temporary file
        # Because we used .cache() and .count() above, Spark no longer needs this file.
        if os.path.exists(temp_filename):
            os.remove(temp_filename)
            print(f"Temporary file {temp_filename} deleted.")

# Run the pipeline

In [14]:
# --- EXECUTION LOOP ---
# This simply iterates through your dictionary and calls our self-contained function
for model_key, texts in model_dict.items():
    run_pipeline(model_key, texts, results)

print("\n" + "#"*60)
print(f"WORKFLOW SUCCESSFUL: {len(results)} models cached and ready for visualization.")
print("#"*60)


INITIATING PIPELINE FOR: bert_sequence_classifier_antisemitism
bert_sequence_classifier_antisemitism download started this may take some time.
Approximate size to download 390.8 MB
[ — ]bert_sequence_classifier_antisemitism download started this may take some time.
Approximate size to download 390.8 MB
[ \ ]Download done! Loading the resource.
[OK!]


Inference complete. 6 records materialized in Spark memory.
Success! 6 rows cached in memory for bert_sequence_classifier_antisemitism.
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|text                                                                                                                                                                                                                                                                         |result|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|Shylock in Merchant of Ven

Inference complete. 9 records materialized in Spark memory.
Success! 9 rows cached in memory for bert_sequence_classifier_trec_coarse.
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|text                                                                                                                                                                       |result|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|Germany is the largest country in Europe economically.                                                                                                                     |[LOC] |
|What other prince showed his paintings in a two-prince exhibition with Prince Charles in London?                                            

Inference complete. 7 records materialized in Spark memory.
Success! 7 rows cached in memory for bert_sequence_classifier_hatexplain.
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+
|text                                                                                                                                                                                                                                           |result       |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+
|If English is not imposition then Hindi is also not imposition shut up chutiya retards stop hindi

Inference complete. 20 records materialized in Spark memory.
Success! 20 rows cached in memory for bert_sequence_classifier_emotion.
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|text                                                                                                                                                                                                                                                                                                                                                                                            |result    |
+----------------------------------------------------------------------

Inference complete. 6 records materialized in Spark memory.
Success! 6 rows cached in memory for bert_sequence_classifier_banking77.
+--------------------------------------------------------------------+-----------------------------+
|text                                                                |result                       |
+--------------------------------------------------------------------+-----------------------------+
|I have been waiting over a week. Is the card still coming?          |[card_arrival]               |
|I need a transaction reversed from my account.                      |[transaction_charged_twice]  |
|How long does it take for cards to be delivered after ordering them?|[card_delivery_estimate]     |
|I've just been married and need to update my name                   |[edit_personal_details]      |
|I'm interested in learning more about disposable virtual cards.\t   |[get_disposable_virtual_card]|
|I tried topping up using my card, but the money is gone?  

# Visualize results

In [15]:
print("\n" + "="*50)
print("VISUALIZING RESULTS (NATIVE SPARK SQL)")
print("="*50)

for model_name, result in results.items():
    # 1. Explode and zip the Spark NLP columns
    # We rename 'metadata' to 'meta' for easier SQL access
    res = result.select(F.explode(F.arrays_zip(
        result.document.result, 
        result.pred_class.result,
        result.pred_class.metadata
    )).alias("col"))\
    .select(
        F.expr("col['0']").alias("sentence"),
        F.expr("col['1']").alias("prediction"),
        F.expr("col['2']").alias("meta")
    )
                  
    if res.count() >= 1:
        print(f"\n[ MODEL: {model_name} ]")
        
        # 2. THE FIX: Access the map natively using F.expr
        # This replaces the UDF that was causing the Python 3.12 crash.
        # It looks up the prediction string as a key in the meta map.
        res_final = res.select(
            "sentence", 
            "prediction", 
            F.expr("meta[prediction]").alias("confidence")
        )
        
        # 3. Show results
        res_final.show(truncate=False)
        print("\n" + "*"*50)


VISUALIZING RESULTS (NATIVE SPARK SQL)

[ MODEL: bert_sequence_classifier_antisemitism ]
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------+
|sentence                                                                                                                                                                                                                                                                     |prediction|confidence|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------+
|Shylock in Merchant of Veni